In [2]:
import pandas as pd
import duckdb

In [3]:
!ls /kaggle/input

steam-dataset-2025-multi-modal-gaming-analytics


In [4]:
!ls /kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics

steam_dataset_2025_csv_package_v1
steam_dataset_2025_embeddings_package_v1
steam_dataset_2025_power_users_dump_v1
steam-dataset-2025-v1


In [5]:
BASE_PATH = (
    "/kaggle/input/"
    "steam-dataset-2025-multi-modal-gaming-analytics/"
    "steam_dataset_2025_csv_package_v1/"
    "steam_dataset_2025_csv"
)

import os
os.listdir(BASE_PATH)

['application_platforms.csv',
 'application_publishers.csv',
 'application_genres.csv',
 'MANIFEST.json',
 'genres.csv',
 'categories.csv',
 'reviews.csv',
 'application_categories.csv',
 'developers.csv',
 'applications.csv',
 'publishers.csv',
 'platforms.csv',
 'application_developers.csv']

In [6]:
import pandas as pd

applications = pd.read_csv(f"{BASE_PATH}/applications.csv")
applications.shape

/tmp/ipykernel_55/1212156742.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  applications = pd.read_csv(f"{BASE_PATH}/applications.csv")


(239664, 30)

In [7]:
applications.head()

,appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,...,mat_pc_os_min,mat_pc_processor_min,mat_pc_memory_min,mat_pc_graphics_min,mat_pc_os_rec,mat_pc_processor_rec,mat_pc_memory_rec,mat_pc_graphics_rec,created_at,updated_at
0,10,Counter-Strike,game,False,2000-11-01,0,Play the world's number 1 online action game. ...,"English<strong>*</strong>, French<strong>*</st...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
1,20,Team Fortress Classic,game,False,1999-04-01,0,One of the most popular online action games of...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
2,30,Day of Defeat,game,False,2003-05-01,0,Enlist in an intense brand of Axis vs. Allied ...,"English, French, German, Italian, Spanish - Spain",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
3,40,Deathmatch Classic,game,False,2001-06-01,0,Enjoy fast-paced multiplayer gaming with Death...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
4,50,Half-Life: Opposing Force,game,False,1999-11-01,0,Return to the Black Mesa Research Facility as ...,"English, French, German, Korean",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00


In [8]:
applications.columns

Index(['appid', 'name', 'type', 'is_free', 'release_date', 'required_age',
       'short_description', 'supported_languages', 'header_image',
       'background', 'metacritic_score', 'recommendations_total',
       'mat_supports_windows', 'mat_supports_mac', 'mat_supports_linux',
       'mat_initial_price', 'mat_final_price', 'mat_discount_percent',
       'mat_currency', 'mat_achievement_count', 'mat_pc_os_min',
       'mat_pc_processor_min', 'mat_pc_memory_min', 'mat_pc_graphics_min',
       'mat_pc_os_rec', 'mat_pc_processor_rec', 'mat_pc_memory_rec',
       'mat_pc_graphics_rec', 'created_at', 'updated_at'],
      dtype='object')

In [9]:
con = duckdb.connect()

In [10]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/application_genres.csv')
LIMIT 5
""").df()

,appid,genre_id
0,10,122
1,20,122
2,30,122
3,40,122
4,50,122


In [11]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/genres.csv')
LIMIT 5
""").df()

,id,name
0,1,Aventură
1,2,Многопользовательские игры
2,3,Nezávislé
3,4,Strategie
4,5,Strategy


In [12]:
con.execute(f"""
SELECT
    ag.appid,
    string_agg(g.name, ', ') AS genres
FROM read_csv_auto('{BASE_PATH}/application_genres.csv') ag
JOIN read_csv_auto('{BASE_PATH}/genres.csv') g
    ON ag.genre_id = g.id
GROUP BY ag.appid
LIMIT 5
""").df()

,appid,genres
0,1502,"Strategy, Indie"
1,2850,"Strategy, Simulation"
2,3612,Casual
3,3810,Action
4,6020,Action


In [13]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_genres AS
SELECT
    ag.appid,
    string_agg(g.name, ', ') AS genres
FROM read_csv_auto('{BASE_PATH}/application_genres.csv') ag
JOIN read_csv_auto('{BASE_PATH}/genres.csv') g
    ON ag.genre_id = g.id
GROUP BY ag.appid
""")

In [14]:
con.execute("""
SELECT *
FROM app_genres
LIMIT 5
""").df()

,appid,genres
0,1502,"Strategy, Indie"
1,2850,"Strategy, Simulation"
2,3612,Casual
3,3810,Action
4,6020,Action


In [15]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/application_categories.csv')
LIMIT 5
""").df()

,appid,category_id
0,10,72
1,10,90
2,10,130
3,10,133
4,10,193


In [16]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/categories.csv')
LIMIT 5
""").df()

,id,name
0,1,Remote Play en móvil
1,2,Flera spelare
2,3,Tablette Remote Play
3,4,Multiplayer
4,5,Таблицы лидеров Steam


In [17]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_categories AS
SELECT
    ac.appid,
    string_agg(c.name, ', ') AS categories
FROM read_csv_auto('{BASE_PATH}/application_categories.csv') ac
JOIN read_csv_auto('{BASE_PATH}/categories.csv') c
    ON ac.category_id = c.id
GROUP BY ac.appid
""")

In [18]:
con.execute("""
SELECT *
FROM app_categories
LIMIT 5
""").df()

,appid,categories
0,2458580,"Steam Achievements, Family Sharing, Single-player"
1,2458760,"Single-player, Downloadable Content"
2,2459190,"Steam Achievements, Family Sharing, Single-player"
3,2459270,"Family Sharing, Single-player"
4,2459820,"Family Sharing, Single-player, Partial Control..."


In [19]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_developers AS
SELECT
    ad.appid,
    string_agg(DISTINCT d.name, ', ') AS developers
FROM read_csv_auto('{BASE_PATH}/application_developers.csv') ad
JOIN read_csv_auto('{BASE_PATH}/developers.csv') d
    ON ad.developer_id = d.id
GROUP BY ad.appid
""")

In [20]:
con.execute("""
SELECT *
FROM app_developers
LIMIT 5
""").df()

,appid,developers
0,2990,"Jordan Freeman Group, ZOOM Platform Media, Bug..."
1,205073,Gaijin Games
2,207620,Telltale Games
3,214420,Doctor Entertainment AB
4,214510,Traveller's Tales


In [21]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_publishers AS
SELECT
    ap.appid,
    string_agg(DISTINCT p.name, ', ') AS publishers
FROM read_csv_auto('{BASE_PATH}/application_publishers.csv') ap
JOIN read_csv_auto('{BASE_PATH}/publishers.csv') p
    ON ap.publisher_id = p.id
GROUP BY ap.appid
""")

In [22]:
con.execute("""
SELECT *
FROM app_publishers
LIMIT 5
""").df()

,appid,publishers
0,56434,"Feral Interactive (Mac/Linux), SEGA"
1,61520,Paradox Interactive
2,218330,Kiz Studios
3,226120,Digital Eel
4,233110,Kuno Interactive


In [23]:
con.execute(f"""
CREATE OR REPLACE TABLE applications_wide AS
SELECT
    a.*,
    g.genres,
    c.categories,
    d.developers,
    p.publishers
FROM read_csv_auto(
        '{BASE_PATH}/applications.csv',
        ignore_errors=true
     ) a
LEFT JOIN app_genres g      ON a.appid = g.appid
LEFT JOIN app_categories c  ON a.appid = c.appid
LEFT JOIN app_developers d  ON a.appid = d.appid
LEFT JOIN app_publishers p  ON a.appid = p.appid
""")

In [24]:
con.execute("""
SELECT COUNT(*) FROM applications_wide
""").df()

,count_star()
0,239653


In [25]:
con.execute("""
SELECT * FROM applications_wide LIMIT 5
""").df()

,appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,...,mat_pc_os_rec,mat_pc_processor_rec,mat_pc_memory_rec,mat_pc_graphics_rec,created_at,updated_at,genres,categories,developers,publishers
0,670490,Rise of Man,game,False,2017-09-15,0,Rise of Man is a pre historic strategy game wi...,"English<strong>*</strong>, Simplified Chinese<...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,Windows 7 / Windows 8,Intel Core i5 processor (or greater) 64bit,8 GB RAM,512 MB DirectX 10 compatible card,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:04:07.947948+00:00,"Strategy, Simulation, Indie, Early Access","Family Sharing, Single-player",Darkcross Games,Darkcross Games
1,670500,RC Plane 3,game,True,2017-08-07,0,Learn to fly a large selection of RC Planes in...,English<strong>*</strong><br><strong>*</strong...,https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,None,None,None,None,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:04:14.546455+00:00,"Simulation, Racing, Sports","Steam Achievements, PvP, Cross-Platform Multip...",FrozenPepper S.R.L,FrozenPepper S.R.L.
2,670510,ColorBlend FX: Desaturation,game,False,2024-04-18,0,Help Splatians blend the stolen colors back wi...,"English<strong>*</strong>, Bulgarian, Simplifi...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,Windows 10,i5 6th generation or new,8 GB RAM,nVidia GeForce GTX 1060 and up / AMD RX 580 an...,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:04:45.421414+00:00,"Indie, Adventure, Action","Steam Achievements, Full controller support, F...",Pi-Dev Bulgaria,Pi-Dev Bulgaria
3,670550,Light Biker Hockey,game,False,NaT,0,Do you like motorbikes and hockey too? Than th...,English<strong>*</strong><br><strong>*</strong...,https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,None,None,None,None,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:02:12.010655+00:00,"Racing, Sports, Adventure","PvP, In-App Purchases, Cross-Platform Multipla...",Theodor Niklas,Theodor Niklas
4,670560,3D MiniGolf: Candy Shop,dlc,True,2017-10-09,0,New DLC &quot;Candy Shopf&quot; available! Sug...,"English<strong>*</strong>, German<strong>*</st...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,10.8,2 GHz Intel-based and above,2 GB RAM,256 MB RAM required,2025-09-07 12:36:43.243384+00:00,2025-09-29 06:02:12.010655+00:00,"Simulation, Casual, Sports","Steam Achievements, Family Sharing, Single-pla...",Z-Software GmbH,familyplay


In [26]:
df = con.execute("""
SELECT *
FROM applications_wide
""").df()

df.shape

(239653, 34)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239653 entries, 0 to 239652
Data columns (total 34 columns):
 #   Column                 Non-Null Count   Dtype                  
---  ------                 --------------   -----                  
 0   appid                  239653 non-null  int64                  
 1   name                   239653 non-null  object                 
 2   type                   236986 non-null  object                 
 3   is_free                239653 non-null  bool                   
 4   release_date           202799 non-null  datetime64[us]         
 5   required_age           239653 non-null  int64                  
 6   short_description      224162 non-null  object                 
 7   supported_languages    221995 non-null  object                 
 8   header_image           239653 non-null  object                 
 9   background             239653 non-null  object                 
 10  metacritic_score       5298 non-null    float64         

In [28]:
df.describe(include="all").T.head(34)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
appid,239653.0,NaN,NaN,NaN,2032205.26384,10.0,1122588.0,1993730.0,2946270.0,3996190.0,1068241.097208
name,239653,237753,Aurora,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,236986,5,game,150276,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_free,239653,2,False,191381,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,202799,NaN,NaN,NaN,2021-10-08 08:04:34.510229,1969-12-31 00:00:00,2019-07-12 00:00:00,2022-06-03 00:00:00,2024-06-12 00:00:00,9998-12-31 00:00:00,NaN
required_age,239653.0,NaN,NaN,NaN,0.271376,0.0,0.0,0.0,0.0,120.0,2.115163
short_description,224162,206921,Embark on your adventure with player from all ...,284,NaN,NaN,NaN,NaN,NaN,NaN,NaN
supported_languages,221995,30502,English,54664,NaN,NaN,NaN,NaN,NaN,NaN,NaN
header_image,239653,239565,https://shared.akamai.steamstatic.com/store_it...,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
background,239653,239653,https://store.akamai.steamstatic.com/images/st...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
